## **ЗПАД | Лабораторна робота 2**
**Виконав: Пономаренко Роман Володимирович | ФБ-44**

### **Частина 1**

Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних;

In [1]:
import urllib.request
import os
from datetime import datetime
from IPython.display import display

def download_vhi():
    if not os.path.exists("vhi_data"):
        os.makedirs("vhi_data")
        
    now = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    for province_id in range(1, 28):
        if any(f.startswith(f"vhi_id_{province_id}_") for f in os.listdir("vhi_data")):
            print(f"Область {province_id} вже завантажена.")
            continue
            
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        filename = f"vhi_data/vhi_id_{province_id}_{now}.csv"
        
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req) as response, open(filename, 'wb') as out_file:
                out_file.write(response.read())
            print(f"Завантажено: {filename}")
        except Exception as e:
            print(f"Помилка завантаження {province_id}: {e}")

download_vhi()

Область 1 вже завантажена.
Область 2 вже завантажена.
Область 3 вже завантажена.
Область 4 вже завантажена.
Область 5 вже завантажена.
Область 6 вже завантажена.
Область 7 вже завантажена.
Область 8 вже завантажена.
Область 9 вже завантажена.
Область 10 вже завантажена.
Область 11 вже завантажена.
Область 12 вже завантажена.
Область 13 вже завантажена.
Область 14 вже завантажена.
Область 15 вже завантажена.
Область 16 вже завантажена.
Область 17 вже завантажена.
Область 18 вже завантажена.
Область 19 вже завантажена.
Область 20 вже завантажена.
Область 21 вже завантажена.
Область 22 вже завантажена.
Область 23 вже завантажена.
Область 24 вже завантажена.
Область 25 вже завантажена.
Область 26 вже завантажена.
Область 27 вже завантажена.


Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області (див. How to Handle Missing Data in Python?)


In [2]:
import pandas as pd
import os
from io import StringIO

def load_and_clean_data(folder="vhi_data"):
    dfs = []
    for filename in os.listdir(folder):
        if not filename.endswith(".csv"): continue
        
        noaa_id = int(filename.split('_')[2])
        
        with open(os.path.join(folder, filename), 'r') as file:
            lines = file.readlines()
            
        clean_lines = []
        for line in lines:
            line = line.replace('<tt><pre>', '').replace('</pre></tt>', '').replace('<br>', '').strip()
            if line.endswith(','):
                line = line[:-1]
            if line and line[0].isdigit():
                clean_lines.append(line)
        
        if not clean_lines:
            continue
            
        csv_data = StringIO('\n'.join(clean_lines))
        df = pd.read_csv(csv_data, sep=r'\s*,\s*', engine='python', header=None, 
                         names=['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'])
        
        df['NOAA_ID'] = noaa_id
        
        df['year'] = pd.to_numeric(df['year'], errors='coerce')
        df['VHI'] = pd.to_numeric(df['VHI'], errors='coerce')
        
        df = df[df['VHI'] != -1]
        df.dropna(subset=['year', 'VHI'], inplace=True)
        
        dfs.append(df)
        
    return pd.concat(dfs, ignore_index=True)

df_raw = load_and_clean_data()
print("Дані після очищення (з оригінальними індексами NOAA):")
display(df_raw.head())

Дані після очищення (з оригінальними індексами NOAA):


,year,week,SMN,SMT,VCI,TCI,VHI,NOAA_ID
0,1982,1,0.053,260.31,45.01,39.46,42.23,1
1,1982,2,0.054,262.29,46.83,31.75,39.29,1
2,1982,3,0.055,263.82,48.13,27.24,37.68,1
3,1982,4,0.053,265.33,46.09,23.91,35.00,1
4,1982,5,0.050,265.66,41.46,26.65,34.06,1


Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). 


In [3]:
def change_province_indices(df):
    province_dict = {
        1: (22, "Черкаська"), 2: (24, "Чернігівська"), 3: (23, "Чернівецька"), 
        4: (25, "АР Крим"), 5: (3, "Дніпропетровська"), 6: (4, "Донецька"), 
        7: (8, "Івано-Франківська"), 8: (19, "Харківська"), 9: (20, "Херсонська"), 
        10: (21, "Хмельницька"), 11: (9, "Київська"), 12: (26, "м. Київ"), 
        13: (10, "Кіровоградська"), 14: (11, "Луганська"), 15: (12, "Львівська"), 
        16: (13, "Миколаївська"), 17: (14, "Одеська"), 18: (15, "Полтавська"), 
        19: (16, "Рівненська"), 20: (27, "м. Севастополь"), 21: (17, "Сумська"), 
        22: (18, "Тернопільська"), 23: (6, "Закарпатська"), 24: (1, "Вінницька"), 
        25: (2, "Волинська"), 26: (7, "Запорізька"), 27: (5, "Житомирська")
    }
    
    df['Province_ID'] = df['NOAA_ID'].map(lambda x: province_dict.get(x, (x, "Unknown"))[0])
    df['Province_Name'] = df['NOAA_ID'].map(lambda x: province_dict.get(x, (x, "Unknown"))[1])
    df.drop(columns=['NOAA_ID'], inplace=True)
    
    return df

df_vhi = change_province_indices(df_raw.copy())
print("Дані після заміни індексів:")
display(df_vhi.head())

Дані після заміни індексів:


,year,week,SMN,SMT,VCI,TCI,VHI,Province_ID,Province_Name
0,1982,1,0.053,260.31,45.01,39.46,42.23,22,Черкаська
1,1982,2,0.054,262.29,46.83,31.75,39.29,22,Черкаська
2,1982,3,0.055,263.82,48.13,27.24,37.68,22,Черкаська
3,1982,4,0.053,265.33,46.09,23.91,35.00,22,Черкаська
4,1982,5,0.050,265.66,41.46,26.65,34.06,22,Черкаська


Реалізувати процедури для формування вибірок наступного виду:

* Ряд VHI для області за вказаний рік;

In [4]:
# Ряд VHI для області за вказаний рік
def get_vhi_for_year(df, province_id, year):
    return df[(df['Province_ID'] == province_id) & (df['year'] == year)][['year', 'week', 'VHI', 'Province_Name']]

# Для першої вибірки (одна область, один рік)
target_prov_1 = 1      # ID області 
target_year_1 = 2020   # Рік

name_1 = df_vhi[df_vhi['Province_ID'] == target_prov_1]['Province_Name'].iloc[0] if not df_vhi[df_vhi['Province_ID'] == target_prov_1].empty else "Невідома"
print(f"Ряд VHI для області: {name_1} (ID {target_prov_1}) за {target_year_1} рік (перші 5 записів):")
display(get_vhi_for_year(df_vhi, target_prov_1, target_year_1).head())

Ряд VHI для області: Вінницька (ID 1) за 2020 рік (перші 5 записів):


,year,week,VHI,Province_Name
52204,2020,1,40.92,Вінницька
52205,2020,2,43.19,Вінницька
52206,2020,3,44.74,Вінницька
52207,2020,4,45.29,Вінницька
52208,2020,5,44.80,Вінницька


* Ряд VHI за вказаний діапазон років для вказаних областей;

In [5]:
# Ряд VHI за вказаний діапазон років для вказаних областей
def get_vhi_for_range(df, province_ids, year_start, year_end):
    return df[(df['Province_ID'].isin(province_ids)) & (df['year'].between(year_start, year_end))][['year', 'week', 'VHI', 'Province_Name']]

# Для другої та третьої вибірок (кілька областей, діапазон років)
target_provs_list = [1, 9]  # Список ID областей
target_year_start = 2010    # Початковий рік
target_year_end = 2012      # Кінцевий рік
names_list = ", ".join(df_vhi[df_vhi['Province_ID'].isin(target_provs_list)]['Province_Name'].unique())

print(f"\nРяд VHI для областей: {names_list} (ID {target_provs_list}) за {target_year_start}-{target_year_end} роки (перші 5 записів):")
display(get_vhi_for_range(df_vhi, target_provs_list, target_year_start, target_year_end).head())


Ряд VHI для областей: Київська, Вінницька (ID [1, 9]) за 2010-2012 роки (перші 5 записів):


,year,week,VHI,Province_Name
23266,2010,1,47.84,Київська
23267,2010,2,47.23,Київська
23268,2010,3,49.70,Київська
23269,2010,4,52.06,Київська
23270,2010,5,52.79,Київська


* Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани;

In [6]:
# Пошук екстремумів (min та max), середнього, медіани
def get_extremes(df, province_ids, year_start, year_end):
    filtered_df = get_vhi_for_range(df, province_ids, year_start, year_end)
    vhi_data = filtered_df['VHI']
    
    if vhi_data.empty:
        return "Немає даних для такого запиту"
        
    return {
        "Min": float(vhi_data.min()),
        "Max": float(vhi_data.max()),
        "Mean": round(float(vhi_data.mean()), 2), 
        "Median": float(vhi_data.median())
    }

target_provs_list = [6, 12]  # Список ID областей
target_year_start = 2013    # Початковий рік
target_year_end = 2016      # Кінцевий рік
names_list = ", ".join(df_vhi[df_vhi['Province_ID'].isin(target_provs_list)]['Province_Name'].unique())

print(f"\nЕкстремуми та статистика для областей: {names_list} за {target_year_start}-{target_year_end} роки:")
print(get_extremes(df_vhi, target_provs_list, target_year_start, target_year_end))


Екстремуми та статистика для областей: Львівська, Закарпатська за 2013-2016 роки:
{'Min': 32.81, 'Max': 67.6, 'Mean': 49.31, 'Median': 49.295}
